# 1. Diagnóstico sobre Bronze
Primera mirada a los datos sin retocar: dimensiones, tipos, muestra y faltantes.

In [ ]:
# 1) Importamos las librerías con sus alias convencionales
import matplotlib.pyplot as plt  # gráficos base (Seaborn se apoya en él)
import pandas as pd  # tablas (DataFrame)
import seaborn as sns  # gráficos estadísticos de alto nivel

from flows.common.paths import BRONZE, BRONZE_LABELED_NAME, SILVER, SILVER_NAME

# 2) Estilo global: todos los gráficos heredan esta configuración
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.titlesize"] = 13

# 3) Cargamos Bronze (nunca la URL: la ingesta ya la guardó)
df = pd.read_csv(BRONZE / BRONZE_LABELED_NAME)

## Paso 1. Dimensiones
¿Cuántos pasajeros (filas) y variables (columnas) hay?

In [ ]:
# shape es un ATRIBUTO: devuelve (filas, columnas)
df.shape

## Paso 2. Variables y tipos
Comparamos los no nulos de cada columna con el total de filas (891) para detectar faltantes.

In [ ]:
# info() es un MÉTODO: imprime nombre, no nulos y tipo de cada columna
df.info()

## Paso 3. Primeras 10 filas

In [ ]:
df.head(10)

## Paso 4. Estadísticas descriptivas

In [ ]:
# Columnas numéricas: conteo, media, desviación, mínimo, cuartiles, máximo
df.describe()

In [ ]:
# Columnas NO numéricas: conteo, valores únicos, más frecuente (top) y su frecuencia
df.select_dtypes(exclude="number").describe()

## Paso 5. Datos faltantes

In [ ]:
# Conteo de nulos por columna (True cuenta como 1)
df.isnull().sum()

In [ ]:
# Mapa de calor: cada línea amarilla es un dato faltante
sns.heatmap(df.isnull(), cbar=False, yticklabels=False, cmap="viridis")
plt.title("Datos faltantes por columna (Bronze)")
plt.show()

# 2. Conociendo a los pasajeros (Silver)
A partir de aquí el cuaderno lee la capa Silver.

In [ ]:
# Recargamos desde Silver (incluye la columna Person creada en el módulo)
df = pd.read_parquet(SILVER / SILVER_NAME)
df.shape

## Paso 6. Distribución por género y su cruce con la clase

In [ ]:
sns.countplot(data=df, x="Sex")
plt.title("Pasajeros por género")
plt.xlabel("Género")
plt.ylabel("Pasajeros")
plt.show()

In [ ]:
sns.countplot(data=df, x="Sex", hue="Pclass")
plt.title("Género por clase")
plt.xlabel("Género")
plt.ylabel("Pasajeros")
plt.show()

In [ ]:
# Proporción de cada género DENTRO de cada clase (cada columna suma 1)
pd.crosstab(df["Sex"], df["Pclass"], normalize="columns").round(3)

## Paso 7. Distribución de edades
Age todavía no está imputada en Silver: solo vemos pasajeros con edad conocida.

In [ ]:
sns.histplot(df["Age"].dropna(), bins=30)
plt.title("Distribución de edades")
plt.xlabel("Edad (años)")
plt.ylabel("Pasajeros")
plt.show()

In [ ]:
# Edad promedio (Pandas ignora los NaN)
df["Age"].mean()

In [ ]:
# Mediana, mínimo y máximo
df["Age"].median(), df["Age"].min(), df["Age"].max()

## Paso 8. Variable Person (hombre, mujer o niño)
Creada en `flows/01_data_engineering.py` → `add_person`. Un pasajero sin edad nunca es `child`: conserva su sexo.

In [ ]:
df["Person"].value_counts()

### Reto 8.1. ¿Cuántos niños "faltan" por edad desconocida?

In [ ]:
n_sin_edad = df["Age"].isnull().sum()

# Proporción de niños entre quienes SÍ tienen edad
con_edad = df[df["Age"].notna()]
prop_ninos = (con_edad["Person"] == "child").mean()

ninos_faltantes = n_sin_edad * prop_ninos
print(f"Pasajeros sin edad: {n_sin_edad}")
print(f"Proporción de niños entre los que tienen edad: {prop_ninos:.3f}")
print(f'Niños que probablemente "faltan": {ninos_faltantes:.0f}')

## Paso 9. ¿Cuántos niños viajaban en cada clase?

In [ ]:
pd.crosstab(df["Pclass"], df["Person"])

## Paso 10. Densidad (KDE) de la edad
`common_norm=False` normaliza cada grupo por separado para que los grupos grandes no aplasten a los pequeños.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)
sns.kdeplot(data=df, x="Age", hue="Sex", fill=True, common_norm=False, ax=axes[0])
sns.kdeplot(data=df, x="Age", hue="Person", fill=True, common_norm=False, ax=axes[1])
sns.kdeplot(
    data=df, x="Age", hue="Pclass", fill=True, common_norm=False, palette="viridis", ax=axes[2]
)
axes[0].set_title("Edad por género")
axes[1].set_title("Edad por tipo de persona")
axes[2].set_title("Edad por clase")
plt.tight_layout()
plt.show()

# 3. Ubicación, embarque y composición familiar

In [ ]:
# Recargamos Silver para tener las columnas nuevas (TCabin, HasCabin, Familia, Alone, Survivor)
df = pd.read_parquet(SILVER / SILVER_NAME)
df.shape

## Paso 11. Cubierta de la cabina
`TCabin` es la primera letra de `Cabin`; `X` = cabina desconocida.

In [ ]:
df["TCabin"].value_counts()

## Paso 12. Pasajeros por cubierta (con depuración de anómalos)

In [ ]:
# Solo cabinas conocidas
cubierta = df.loc[df["TCabin"] != "X", "TCabin"]

sns.countplot(x=cubierta, order=sorted(cubierta.unique()))
plt.title("Pasajeros con cabina por cubierta")
plt.xlabel("Cubierta")
plt.ylabel("Pasajeros")
plt.show()

In [ ]:
# La cubierta 'T' tiene un solo registro: se excluye SOLO del gráfico, no de df
cubierta = cubierta[cubierta != "T"]

sns.countplot(x=cubierta, order=sorted(cubierta.unique()))
plt.title("Pasajeros con cabina por cubierta (sin anómalos)")
plt.xlabel("Cubierta")
plt.ylabel("Pasajeros")
plt.show()

## Paso 13. Puerto de embarque y clase

In [ ]:
sns.countplot(data=df, x="Embarked", hue="Pclass", order=["S", "C", "Q"])
plt.title("Puerto de embarque por clase")
plt.xlabel("Puerto")
plt.ylabel("Pasajeros")
plt.show()

In [ ]:
# Composición de clases de cada puerto (cada fila suma 1)
pd.crosstab(df["Embarked"], df["Pclass"], normalize="index").round(3)

## Paso 14. Variable Alone (solo vs. con familia)
Creada en el módulo → `add_family_features`: `Familia = SibSp + Parch`.

In [ ]:
df["Alone"].value_counts(normalize=True)

## Paso 15. Distribución de Alone

In [ ]:
sns.countplot(data=df, x="Alone")
plt.title("Pasajeros solos vs. con familia")
plt.xlabel("")
plt.ylabel("Pasajeros")
plt.show()

# 4. Análisis de supervivencia
La media de `Survived` (0/1) es la tasa de supervivencia. Las barras verticales de los gráficos de puntos son intervalos de confianza al 95 %.

## Paso 16. Panorama general

In [ ]:
sns.countplot(data=df, x="Survivor", order=["No", "Yes"])
plt.title("Supervivientes vs. no supervivientes")
plt.xlabel("¿Sobrevivió?")
plt.ylabel("Pasajeros")
plt.show()

In [ ]:
# Tasa general de supervivencia (debería rondar 0.38)
df["Survived"].mean()

## Paso 17. Supervivencia por clase

In [ ]:
g = sns.catplot(data=df, x="Pclass", y="Survived", kind="point", height=4, aspect=1.4)
g.set_axis_labels("Clase", "Tasa de supervivencia")
plt.title("Supervivencia por clase")
plt.show()

## Paso 18. Supervivencia por clase y tipo de persona

In [ ]:
g = sns.catplot(
    data=df,
    x="Pclass",
    y="Survived",
    hue="Person",
    kind="point",
    hue_order=["child", "female", "male"],
    height=4,
    aspect=1.5,
)
g.set_axis_labels("Clase", "Tasa de supervivencia")
plt.title("Supervivencia por clase y tipo de persona")
plt.show()

## Paso 19. Edad y supervivencia
Con `y` binaria, la recta es solo una aproximación visual; `x_bins` muestra la tasa media por grupo de edad.

In [ ]:
sns.lmplot(data=df, x="Age", y="Survived")
plt.title("Supervivencia según la edad")
plt.show()

In [ ]:
sns.lmplot(data=df, x="Age", y="Survived", hue="Pclass")
plt.title("Supervivencia según la edad, por clase")
plt.show()

In [ ]:
generaciones = [10, 20, 40, 60, 80]
sns.lmplot(data=df, x="Age", y="Survived", hue="Pclass", x_bins=generaciones)
plt.title("Tasa de supervivencia por generación y clase")
plt.show()

In [ ]:
sns.lmplot(data=df, x="Age", y="Survived", hue="Sex", x_bins=generaciones)
plt.title("Tasa de supervivencia por generación y género")
plt.show()

## Paso 20. Relación tarifa-edad

In [ ]:
plt.scatter(df["Age"], df["Fare"], alpha=0.5)
plt.title("Tarifa vs. edad")
plt.xlabel("Edad (años)")
plt.ylabel("Tarifa")
plt.show()

In [ ]:
sns.scatterplot(data=df, x="Age", y="Fare", hue="Pclass", alpha=0.5, palette="viridis")
plt.title("Tarifa vs. edad por clase")
plt.xlabel("Edad (años)")
plt.ylabel("Tarifa")
plt.show()

# 5. Limpieza y preparación de datos
Un dato faltante puede (a) eliminarse, (b) imputarse o (c) conservarse como categoría propia. **Ninguna opción es neutral.**
La limpieza se hizo en `flows/01_data_engineering.py`; aquí verificamos su efecto comparando Bronze (original) con Silver (limpio).

In [ ]:
# Imports adicionales para las Partes 5 a 9
import importlib

from flows.common.paths import SILVER_FINAL_NAME

agg = importlib.import_module("flows.03_aggregation")  # se usa desde la Parte 6

bronze = pd.read_csv(BRONZE / BRONZE_LABELED_NAME)  # datos originales
df = pd.read_parquet(SILVER / SILVER_NAME)  # datos limpios
df.shape

## Paso 21. Datos faltantes en `Embarked`
`impute_embarked` rellena con la moda y `encode_categoricals` crea `NEmbarked` (S → 0, C → 1, Q → 2).

In [ ]:
print("Faltantes en Embarked (Bronze):", bronze["Embarked"].isnull().sum())
print("Moda de Embarked:", bronze["Embarked"].mode()[0])
print("Faltantes en Embarked (Silver):", df["Embarked"].isnull().sum())

In [ ]:
# Verificación de la codificación
pd.crosstab(df["Embarked"], df["NEmbarked"])

In [ ]:
print("Edades faltantes (AgeOriginal):", df["AgeOriginal"].isnull().sum())
print("Edades faltantes (Age imputada):", df["Age"].isnull().sum())
print("Filas imputadas:", df["AgeImputada"].sum())

In [ ]:
# Promedio de edad por género: el valor con el que se rellenó cada grupo
df.groupby("Sex")["AgeOriginal"].mean()

### Reto 22.1. ¿Qué le pasa a la edad al imputar?

In [ ]:
print(f"Desviación estándar ANTES de imputar:   {df['AgeOriginal'].std():.2f}")
print(f"Desviación estándar DESPUÉS de imputar: {df['Age'].std():.2f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
sns.histplot(df["AgeOriginal"].dropna(), bins=30, ax=axes[0])
axes[0].set_title("Edad original (sin imputar)")
sns.histplot(df["Age"], bins=30, ax=axes[1])
axes[1].set_title("Edad imputada por género")
for ax in axes:
    ax.set_xlabel("Edad (años)")
    ax.set_ylabel("Pasajeros")
plt.tight_layout()
plt.show()

## Paso 23. `Sex` a número
`NSex`: female → 0, male → 1.

In [ ]:
pd.crosstab(df["Sex"], df["NSex"])

## Paso 24. Tratamiento de `Cabin`
`NCabin` usa un mapa **fijo** sobre `DECKS` (A=0 … X=8), así los números no cambian si una cubierta desaparece de los datos.

In [ ]:
pd.crosstab(df["TCabin"], df["NCabin"])

## Paso 25. Títulos extraídos del nombre

In [ ]:
titulos_originales = df["Name"].str.extract(r",\s*([^.]+)\.", expand=False).str.strip()
titulos_originales.value_counts()

In [ ]:
pd.crosstab(titulos_originales, df["Sex"])

## Paso 26. Títulos consolidados

In [ ]:
df["TTitulo"].value_counts()

In [ ]:
pd.crosstab(df["TTitulo"], df["NTitulo"])

# 6. Correlaciones y estadísticas por grupos
La correlación de Pearson mide relación **lineal** (−1 a +1), no implica causalidad y es poco informativa para categóricas codificadas arbitrariamente.

## Paso 27. Matriz de correlación

In [ ]:
cols = [
    "Survived",
    "Pclass",
    "NSex",
    "Age",
    "SibSp",
    "Parch",
    "Fare",
    "NEmbarked",
    "NCabin",
    "NTitulo",
]
corr = df[cols].corr()

plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, vmin=-1, vmax=1)
plt.title("Matriz de correlación")
plt.show()

In [ ]:
# Correlación de cada variable con la supervivencia, ordenada
corr["Survived"].sort_values()

## Paso 28. Estadísticas por grupos
Calculadas por `group_stats` en `flows/03_aggregation.py` (capa Gold).

In [ ]:
tablas = agg.group_stats(df)
tablas["age_by_survival"]

In [ ]:
tablas["survival_by_sex"]

In [ ]:
tablas["survival_by_pclass"]

In [ ]:
tablas["survival_by_alone"]

## Paso 29. ¿La diferencia por sexo es significativa? (chi-cuadrado)
- **H₀:** `Sex` y `Survived` son independientes.
- **H₁:** sí hay asociación.

Si `p < 0.05` se rechaza H₀.

In [ ]:
pd.crosstab(df["Sex"], df["Survived"])

In [ ]:
res = agg.chi2_sex_survival(df)
print(f"chi2 = {res['chi2']:.1f}, p = {res['p_value']:.2e}, gl = {res['dof']}")

# 7. Visualizaciones comparativas avanzadas

## Paso 30. Boxplots múltiples (2×2)
Se usa `AgeOriginal` para ver solo edades reales (la imputada concentraría valores en los promedios).

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
sns.boxplot(data=df, x="Sex", y="AgeOriginal", ax=axes[0, 0])
sns.boxplot(data=df, x="Pclass", y="AgeOriginal", ax=axes[0, 1])
sns.boxplot(data=df, x="Embarked", y="AgeOriginal", ax=axes[1, 0])
sns.boxplot(data=df, x="Survived", y="AgeOriginal", ax=axes[1, 1])
axes[0, 0].set_title("Edad por género")
axes[0, 1].set_title("Edad por clase")
axes[1, 0].set_title("Edad por puerto")
axes[1, 1].set_title("Edad por supervivencia")
for ax in axes.flat:
    ax.set_ylabel("Edad (años)")
plt.tight_layout()
plt.show()

## Paso 31. Distribución de la tarifa

In [ ]:
sns.histplot(df["Fare"], kde=True, bins=40)
plt.title("Distribución de la tarifa")
plt.xlabel("Tarifa")
plt.ylabel("Pasajeros")
plt.show()

In [ ]:
print(f"Media:     {df['Fare'].mean():.2f}")
print(f"Mediana:   {df['Fare'].median():.2f}")
print(f"Asimetría: {df['Fare'].skew():.2f}")

# 8. Dataset final y verificación
`drop_original_columns` genera `titanic_final.parquet` sin `Name`, `Sex`, `Embarked` ni `Cabin`; `validate_silver` hace fallar el pipeline si quedan nulos en columnas clave.

## Paso 32. Vista final (solo columnas derivadas/numéricas)

In [ ]:
final = pd.read_parquet(SILVER / SILVER_FINAL_NAME)
print(final.shape)
final.columns.tolist()

## Paso 33. Verificación
`AgeOriginal` es la única columna que puede tener nulos (es la edad sin imputar, a propósito).

In [ ]:
final.isnull().sum()

In [ ]:
final.info()

In [ ]:
final.describe()

# 9. Análisis crítico: interacciones, asociaciones y atípicos
Ya no basta con un número: hay que preguntar si es **grande**, si es **confiable** (¿cuántos pasajeros hay detrás?) y si es **justo** compararlo (¿hay una variable de confusión?).
Cada tabla viene de una función de `flows/03_aggregation.py` (capa Gold).

## Paso 34. Interacción clase × género

In [ ]:
agg.survival_by_class_sex(df).round(3)

## Paso 35. Puerto de embarque y supervivencia (¿confusión?)

In [ ]:
agg.survival_by_embarked(df).round(3)

In [ ]:
# Estratificado por clase: si dentro de cada clase las diferencias entre puertos se achican,
# el "efecto puerto" era en realidad un efecto de la clase
agg.survival_by_embarked_class(df).round(2)

## Paso 36. Tarifa y supervivencia dentro de cada clase
`FareQ` = cuartil de la tarifa (0 = más barato, 3 = más caro) calculado **dentro** de cada clase.

In [ ]:
agg.survival_by_fare_quartile_class(df).round(2)

## Paso 37. Tamaño de la familia

In [ ]:
agg.survival_by_family(df).round(3)

In [ ]:
sns.pointplot(data=df, x="Familia", y="Survived")
plt.title("Supervivencia según número de familiares a bordo")
plt.xlabel("Familiares a bordo")
plt.ylabel("Tasa de supervivencia")
plt.show()

## Paso 38. Pasajeros atípicos (regla IQR)
Atípico **no** significa erróneo: antes de eliminar, pregunta si es un error de captura o un caso real.

In [ ]:
agg.outlier_summary(df).round(3)

In [ ]:
# Pasajeros con la tarifa más alta
df.nlargest(10, "Fare")[["Name", "Pclass", "Fare", "TCabin", "Survived"]]